In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
import sys
import random
from statistics import variance
import seaborn
import matplotlib.pyplot as plt

In [ ]:

###### Some functions

def coordinate(i,j,GridNumber):
    x = 1/(GridNumber)*(i-1)
    y = 1/(GridNumber)*(j-1)
    return x, y

def count_duplicates(row):
    value_counts = row.value_counts()
    return [value for value in value_counts.index]

def build_index_sets(row):
    index_sets = {}
    for index, value in enumerate(row):
        if value not in index_sets:
            index_sets[value] = []
        index_sets[value].append(index)
    return index_sets

def coords_to_flat_index(a, b, grid_size):
    """
    Convert 2D coordinates (a, b) to flat 1D index for a 21x21 grid.
    
    Args:
        a: row index (0-20)
        b: column index (0-20)
        grid_size: size of the square grid
    
    Returns:
        flat_index: 1D index (0-440)
    """
    return a * grid_size + b



## Construct the demand matrix

In [ ]:
###### Construct the matrix

demand_grid_number = 10
hub_grid_number = 20
Totalofpair = (pow(demand_grid_number+1,2)+1)*pow(demand_grid_number+1,2)/2
print("The total number of pair is:", Totalofpair)
ColumnsName=[f'P{i,j}' for i in range(1, hub_grid_number + 2) for j in range(1, hub_grid_number + 2)]
# print(ColumnsName)

Rowindex = []
Dia_index = []
Dia_counter = 0

for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):
            if l == j:
                Dia_index.append(Dia_counter)

            temp = f'O{i,j}_D{i,l}'
            Rowindex.append(temp)
            Dia_counter = Dia_counter+1

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):
                temp = f'O{i,j}_D{k,l}'
                Rowindex.append(temp)
                Dia_counter = Dia_counter+1


df = pd.DataFrame( index= Rowindex, columns=ColumnsName)


for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):

            
        
            for m in range(1, hub_grid_number + 2):
                for n in range(1, hub_grid_number + 2):

                    Oi,Oj = coordinate(i,j,demand_grid_number)
                    Di,Dj = coordinate(i,l,demand_grid_number)
                    Pi,Pj = coordinate(m,n,hub_grid_number)
                    
                
                    #df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)

                    df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
                    

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):

                for m in range(1, hub_grid_number + 2):
                    for n in range(1, hub_grid_number + 2):

                        Oi,Oj = coordinate(i,j,demand_grid_number)
                        Di,Dj = coordinate(k,l,demand_grid_number)
                        Pi,Pj = coordinate(m,n,hub_grid_number)

                        #df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)
                        df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
                        
                        
    
    ## Print reduced Arc
                        
# print("the Dia_index is",Dia_index)



# Apply the function to each row
# print(df.head())
result = df.apply(count_duplicates, axis=1)

#print(result)

# print(result.head())

NUnique = 0

for i in range(len(result)):
    NUnique = NUnique + len(result.iloc[i])

print("total number of unique value is", NUnique)
TotalArc = Totalofpair* len(ColumnsName)
print("Total number of arc is:", TotalArc)

print("the percentage of Arc after reduction", NUnique/TotalArc)

###### generate some parameters

rows_index_sets = df.apply(build_index_sets, axis=1)

## Build Kmax

Kmax = []
for i in range(len(rows_index_sets)):
    Kmax.append(len(rows_index_sets[i]))

#print("Kmax",Kmax)

## Calculate distance d_ik

def distance_calculation (data_frame):
    distance={}
    for i in range (0, len(data_frame)):
        k_value = list(sorted(data_frame[i].keys()))
        for k in range (len(data_frame[i])):
            distance[(i,k)] = k_value[k] 
    return distance

distance = distance_calculation(rows_index_sets)

## Get S_ik
def Set_obtain (data_frame):
    Set_ties={}
    for i in range (0, len(data_frame)):
        sorted_dict = dict(sorted(data_frame[i].items()))
        index = list(sorted(data_frame[i].keys()))
        for k in range (len(data_frame[i])):            
            Set_ties[(i,k)] = sorted_dict[index[k]]
    return Set_ties

Set_ties = Set_obtain(rows_index_sets)



In [ ]:
print(len(df))
print(df.iloc[:, 440])

col1 = df[df.columns[0]]  # First column
col2 = df[df.columns[1]]  # Second column
min_sum = np.minimum(col1, col2).sum()
print(f"Sum using df.columns[index]: {min_sum}")

## Warm_start

In [ ]:

###### Define the warm_start function
def warm_start(demand_grid_number,hub_grid_number,Pnumber):
    ## Construct the matrix
    #Totalofpair = (pow(demand_grid_number+1,2)+1)*pow(demand_grid_number+1,2)/2
    #print("The total number of pair is:", Totalofpair)
    ColumnsName=[f'P{i,j}' for i in range(1, hub_grid_number + 2) for j in range(1, hub_grid_number + 2)]
    # print(ColumnsName)

    Rowindex = []
    Dia_index = []
    Dia_counter = 0

    for i in range(1, demand_grid_number + 2):
        for j in range(1, demand_grid_number + 2):
            for l in range(j,demand_grid_number+2):
                if l == j:
                    Dia_index.append(Dia_counter)

                temp = f'O{i,j}_D{i,l}'
                Rowindex.append(temp)
                Dia_counter = Dia_counter+1

            for k in range(i+1,demand_grid_number + 2):
                for l in range(1,demand_grid_number + 2):
                    temp = f'O{i,j}_D{k,l}'
                    Rowindex.append(temp)
                    Dia_counter = Dia_counter+1


    df = pd.DataFrame( index= Rowindex, columns=ColumnsName)


    for i in range(1, demand_grid_number + 2):
        for j in range(1, demand_grid_number + 2):
            for l in range(j,demand_grid_number+2):

                
            
                for m in range(1, hub_grid_number + 2):
                    for n in range(1, hub_grid_number + 2):

                        Oi,Oj = coordinate(i,j,demand_grid_number)
                        Di,Dj = coordinate(i,l,demand_grid_number)
                        Pi,Pj = coordinate(m,n,hub_grid_number)
                        
                    
                        #df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)

                        df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
                        

            for k in range(i+1,demand_grid_number + 2):
                for l in range(1,demand_grid_number + 2):

                    for m in range(1, hub_grid_number + 2):
                        for n in range(1, hub_grid_number + 2):

                            Oi,Oj = coordinate(i,j,demand_grid_number)
                            Di,Dj = coordinate(k,l,demand_grid_number)
                            Pi,Pj = coordinate(m,n,hub_grid_number)

                            #df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)
                            df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
    rows_index_sets = df.apply(build_index_sets, axis=1)

    Kmax = []
    for i in range(len(rows_index_sets)):
        Kmax.append(len(rows_index_sets[i]))

    distance = distance_calculation(rows_index_sets)
    Set_ties = Set_obtain(rows_index_sets)

    


    m = gp.Model()
    m.Params.MIPGap = 0
    m.params.logtoconsole=0

    # variables: t_ij, y_j, 

    Pair_range = range(len(rows_index_sets))
    Location_range = range(len(ColumnsName))

    t = {}
    for i in Pair_range:
        for k in range(Kmax[i]):
            t[i,k] = m.addVar(vtype=GRB.CONTINUOUS,name = f"t_+{i,k}")

    y = m.addVars(Location_range ,vtype=GRB.BINARY, name="y_i")

    Non_Dia_index = [item for item in Pair_range if item not in Dia_index]

    obj = gp.quicksum(

            distance[i,0] + gp.quicksum( (distance[i,k+1]-distance[i,k])*t[i,k] for k in range(Kmax[i]-1))
            for i in Pair_range
        ) + gp.quicksum(

            distance[i,0] + gp.quicksum( (distance[i,k+1]-distance[i,k])*t[i,k] for k in range(Kmax[i]-1))
            for i in Non_Dia_index
        ) 
            

    # Constraints

    m.addConstr(gp.quicksum(y[j] for j in Location_range) == Pnumber,name = "plocation")

    for i in Pair_range:
        m.addConstr(t[i,0]+ gp.quicksum(y[j] for j in Set_ties[i,0]) >= 1,name = f"constraint1_{i}")

    for i in Pair_range:
        for k in range(1,Kmax[i]):
            m.addConstr(t[i,k]+ gp.quicksum(y[j] for j in Set_ties[i,k]) >= t[i,k-1],name = f"constraint2_{i}")



    #print("start the optimization")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()
    runtime = m.Runtime
    yvalues = m.getAttr('x',y)
    print("running time is ", runtime)
    print("The objective value is  ", m.ObjVal)

    hubs = []
    for k,y in y.items():
        if y.x > 0:
            hubs.append(k)

    for i in hubs:
        print(ColumnsName[i])
    

    return yvalues


    

In [ ]:
p_number = 2

print("P_number is:", p_number)
demand_grid_number_warm = 5
hub_grid_number_warm = 50

yvalues_h = warm_start(demand_grid_number_warm,hub_grid_number_warm,p_number)

## Benders Decomposition

In [ ]:

print("########################### warm_start ###########################")
# Set the number and warm start grid
p_number = 2

print("P_number is:", p_number)
demand_grid_number_warm = 5
hub_grid_number_warm = 20


start_time = time.time()
yvalues_h = warm_start(demand_grid_number_warm,hub_grid_number_warm,p_number)




print("-------------- Elapsed %s seconds --------------" % (time.time() - start_time))
print("########################### Phase 0 ###########################")
####### Bender Decomposition
flag = 1 # flag = 1 initialize the algorithm with benders cuts derived from LP relaxatiion
Constraint_reduction = 1 # reduce the unbinding constraints
fixing_variable = 1 # fixing constraint

### generate the critical index for i
def critical_index_i(index,yvalues,Kmax):
    k_i = 0
    count = 0
    val = 1 - sum(yvalues[j] for j in Set_ties[index,count])
    while val > 0 and k_i < Kmax:
        k_i = k_i + 1
        count = count + 1
        val = val - sum(yvalues[j] for j in Set_ties[index,count])
    
    return k_i 


### Main Problem set up

MP = gp.Model()



theta_start = {}
upper_bound = 0

Pair_range = range(len(rows_index_sets))
Location_range = range(len(ColumnsName))
Non_Dia_index = [item for item in Pair_range if item not in Dia_index]

y = MP.addVars(Location_range, lb = 0.0,ub = 1.0, vtype=GRB.CONTINUOUS, name="y_i")
theta = MP.addVars(Pair_range,vtype=GRB.CONTINUOUS, name="theta")
MP.addConstr(gp.quicksum(y[j] for j in Location_range) == p_number,name = "plocation")


## Construct bender cut with heuristic solution

for i in Pair_range:

    k_i = critical_index_i(i,yvalues_h,Kmax[i])
    if i in Non_Dia_index:
        if k_i == 0:
            MP.addConstr(theta[i] >= 2*distance[i,0])
            theta_start[i] =  2*distance[i,0]
            upper_bound += theta_start[i]

        else:
            MP.addConstr(theta[i] >= 2*(distance[i,k_i] - gp.quicksum( 
                        
                        gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i))) 
                        )
            
            theta_start[i] = 2*(distance[i,k_i] - sum(       
                        sum( (distance[i,k_i]-distance[i,h])*yvalues_h[j] for j in Set_ties[i,h] )  for h in range(k_i)))
            
            upper_bound += theta_start[i]



    else:
        if k_i == 0:
            MP.addConstr(theta[i] >= distance[i,0])
            theta_start[i] =  distance[i,0]
            upper_bound += theta_start[i]

        else:
            MP.addConstr(theta[i] >= distance[i,k_i] - gp.quicksum( 
                        gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i))
        
                        )
            
            theta_start[i] = distance[i,k_i] - sum( 
                        
                        sum( (distance[i,k_i]-distance[i,h])*yvalues_h[j] for j in Set_ties[i,h] )  for h in range(k_i))
            upper_bound += theta_start[i]

print("the upper bound of heuristic is :", upper_bound)
                 
print("-------------- Elapsed %s seconds --------------" % (time.time() - start_time))


obj = gp.quicksum(theta[i] for i in Pair_range)


MP.setObjective(obj, GRB.MINIMIZE)



### Begin the MP relaxtion cutting plane until no bender cut found.

iteration = 1

cut_count_total = 0
previous_obj = 0
lower_bound = 0
total_boost = 0

while flag:
    print('================ Iteration ', iteration, ' ===================')
    
    flag = 0
    cut_count = 0
    MP.optimize()
    y_values = MP.getAttr('x',y)
    theta_values = MP.getAttr('x',theta)

    current_obj = MP.ObjVal

    for i in Pair_range:

        k_i = critical_index_i(i,y_values,Kmax[i])
        if i in Non_Dia_index:
            if k_i == 0:
    
                if round(theta_values[i],6)  < round(2*distance[i,0],6):
                    MP.addConstr(theta[i] >= 2*distance[i,0])
                    cut_count += 1
                    flag = 1
            else:
 
                if round(theta_values[i],6) < 2*round(distance[i,k_i] - sum( 
                            
                            sum( (distance[i,k_i]-distance[i,h])*y_values[j] for j in Set_ties[i,h] )  for h in range(k_i)),6):
                    

                    MP.addConstr(theta[i] >= 2*(distance[i,k_i] - gp.quicksum( 
                                gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i)))
                                )
                    cut_count += 1
                    flag = 1
        else:
            if k_i == 0:
                
                if round(theta_values[i],6)  < round(distance[i,0],6):
                    MP.addConstr(theta[i] >= distance[i,0])
                    cut_count += 1
                    flag = 1
            else:

                if round(theta_values[i],6)  < round(distance[i,k_i] - sum( 
                            
                            sum( (distance[i,k_i]-distance[i,h])*y_values[j] for j in Set_ties[i,h] )  for h in range(k_i)),6):
                    
                    MP.addConstr(theta[i] >= distance[i,k_i] - gp.quicksum( 
                                gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i))
                                )
                    cut_count += 1
                    flag = 1
    

    if abs(current_obj-previous_obj) > 100:
        MP.update()
        cut_count_total += cut_count
        print("Current MP relaxation objective value is: ", current_obj)
        print("The number of bender cut in this iteration we found: ", cut_count)
        if iteration > 1:
            print("This iteration boosts the lower bound: ", current_obj-previous_obj )
            total_boost += current_obj-previous_obj

        previous_obj = current_obj
    else:
        
        flag = 0
        
    
    iteration += 1


print("The total boost of lower bound: ", total_boost)
print("The total number of bender cut we found: ", cut_count_total)


### Find the unbinding cosntraint and remove them 
if fixing_variable:
    lower_bound = current_obj
    y_RC = MP.getAttr('RC',y)
    fixing_count = 0
    for i in Location_range:

        if round(lower_bound + y_RC[i],6) > round(upper_bound,6):
            fixing_count += 1
            y[i].ub = 0
        
        if round(lower_bound - y_RC[i],6) > round(upper_bound,6):
            y[i].lb = 1
            fixing_count += 1

    print("The total number of fixing count is :", fixing_count)

    
if Constraint_reduction:
    constra = MP.getConstrs()
    print("The toal number of constraints is: ", len(constra))
    unbinding = [c for c in constra if abs(c.Slack) > 1e-6]

    print("Find total number of unbinding constraints is: ", len(unbinding))

    for item in unbinding:
        MP.remove(item)
    MP.update()

    print("The toal number of constraints after reduction is: ", len(MP.getConstrs()))





### use Gurobi callback function to do Branch-and-bender-cut
print("-------------- Elapsed %s seconds --------------" % (time.time() - start_time))

print("########################### Phase 1 ###########################")

# handle the Integer Programming and set up the heuristic solution start
for i in Location_range:
    y[i].vType = GRB.BINARY
    y[i].Start = yvalues_h[i]

for i in Pair_range:
    theta[i].Start = theta_start[i]


# use Gurobi callback function

def BendersCallback(model, where):
    if where == GRB.Callback.MIPSOL:

        y_values_callback = model.cbGetSolution(y)
        theta_values_callback = model.cbGetSolution(theta)
        #print(theta_values_callback)

        for i in Pair_range:
            k_i = critical_index_i(i,y_values_callback,Kmax[i])
            if i in Non_Dia_index:
                if k_i == 0:
                    
                    if round(theta_values_callback[i],6) < 2*distance[i,0]:
                        
                        model.cbLazy(theta[i] >= 2*distance[i,0])

                        
                else:
                    if round(theta_values_callback[i],6) < 2*(distance[i,k_i] - sum( 
                                
                                sum( (distance[i,k_i]-distance[i,h])*y_values_callback[j] for j in Set_ties[i,h] )  for h in range(k_i))) :
                        
                        model.cbLazy(theta[i] >= 2*(distance[i,k_i] - gp.quicksum( 
                                    gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i)))
                                    )
            else:
                
                if k_i == 0:
                    
                    if round(theta_values_callback[i],6) < distance[i,0] :
                        
                        model.cbLazy(theta[i] >= distance[i,0])            
                else:
                    if round(theta_values_callback[i],6) < distance[i,k_i] - sum( 
                                sum( (distance[i,k_i]-distance[i,h])*y_values_callback[j] for j in Set_ties[i,h] )  for h in range(k_i)):
  
                        model.cbLazy(theta[i] >= distance[i,k_i] - gp.quicksum( 
                                    gp.quicksum( (distance[i,k_i]-distance[i,h])*y[j] for j in Set_ties[i,h] )  for h in range(k_i))
                                    )
                    


### Pass BendersCallback as argument to the optimize function on the main
MP.Params.lazyConstraints = 1
MP.Params.MIPGap = 0
MP.Params.timeLimit = 36000
MP.update()
MP.optimize(BendersCallback)

runtime = MP.Runtime
yvalues = MP.getAttr('x',y)
MP_theta = MP.getAttr('x',theta)
print("running time is ", runtime)
print("The objective value is  ", MP.ObjVal)
hubs = []
for k,y in y.items():
    if y.x > 0:
        hubs.append(k)

for i in hubs:
    print(ColumnsName[i])
    
print("-------------- Elapsed %s seconds --------------" % (time.time() - start_time))




    

## Calculate obj value

In [ ]:
def compute_obj_value(yvalues_h):
    upper_bound = 0
    for i in Pair_range:
        k_i = critical_index_i(i,yvalues_h,Kmax[i])

        if k_i == 0:
            if i in Non_Dia_index:
                upper_bound +=  2*distance[i,0]
            else:
                upper_bound +=  distance[i,0]

        else:
            if i in Non_Dia_index:
                upper_bound += 2*(distance[i,k_i] - sum( 
                        
                        sum( (distance[i,k_i]-distance[i,h])*yvalues_h[j] for j in Set_ties[i,h] )  for h in range(k_i)))
            else:
                upper_bound += distance[i,k_i] - sum( 
                        
                        sum( (distance[i,k_i]-distance[i,h])*yvalues_h[j] for j in Set_ties[i,h] )  for h in range(k_i))
    
    return upper_bound




print(compute_obj_value(yvalues_h))


## K-mediods

In [ ]:
## create new df for objective function calculation
new_df = df.copy()
for idx in range(len(new_df)):
    if idx not in Dia_index:
        new_df.iloc[idx] *= 2


## compute_obj_value
def compute_obj_value(plan, grid_size):
    first_num = coords_to_flat_index(plan[0],plan[1],grid_size+1)
    second_num = coords_to_flat_index(plan[2],plan[3],grid_size+1)
    col1 = new_df[new_df.columns[first_num]]  # First column
    col2 = new_df[new_df.columns[second_num]]  # Second column
    min_column = np.minimum(col1, col2)

    return min_column.sum()

    
    

In [ ]:

class PlanPool:
    def __init__(self):
        self.pool = set()  # Use set for O(1) lookup
    
    def add_plan(self, plan):
        """
        Add a plan to the pool if it's unique
        
        Args:
            plan: list [a, b, c, d]
            
        Returns:
            bool: True if plan was added (unique), False if already exists
        """
        plan_tuple = tuple(plan)  # Convert to tuple (hashable)
        
        if plan_tuple not in self.pool:
            self.pool.add(plan_tuple)
            return True
        return False
    
    def contains_plan(self, plan):
        """Check if plan exists in pool"""
        return tuple(plan) in self.pool
    
    def get_pool_size(self):
        """Get number of unique plans in pool"""
        return len(self.pool)
    
    def get_all_plans(self):
        """Get all plans as list of lists"""
        return [list(plan) for plan in self.pool]
    

# pool = PlanPool()

# # Test adding plans
# plan1 = [5, 5, 10, 10]
# plan2 = [5, 5, 10, 10]  # Duplicate
# plan3 = [3, 7, 15, 2]   # Unique
# plan4 = [3, 7, 15, 2]   # Unique

# print(f"Added plan1: {pool.add_plan(plan1)}")  # True
# print(f"Added plan2: {pool.add_plan(plan2)}")  # False (duplicate)
# print(f"Added plan3: {pool.add_plan(plan3)}")  # True
# print(f"Added plan3: {pool.add_plan(plan4)}")  # True

In [ ]:
def get_plan_neighbors_4directional(plan, grid_size):
    """
    Alternative version using only 4-directional movement (up, down, left, right)
    """
    a, b, c, d = plan
    neighbors = []
    
    # 4-directional movements only
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
    
    # Move first point
    for da, db in directions:
        new_a, new_b = a + da, b + db
        if 0 <= new_a <= grid_size and 0 <= new_b <= grid_size:
            neighbors.append([new_a, new_b, c, d])
    
    # Move second point
    for dc, dd in directions:
        new_c, new_d = c + dc, d + dd
        if 0 <= new_c <= grid_size and 0 <= new_d <= grid_size:
            neighbors.append([a, b, new_c, new_d])
    
    return neighbors


def get_plan_neighbors_8directional(plan, grid_size):
    """
    Alternative version using only 4-directional movement (up, down, left, right)
    """
    a, b, c, d = plan
    neighbors = []
    
    # 8-directional movements only
    directions = [
        (-1, -1), (-1, 0), (-1, 1),  # up-left, up, up-right
        (0, -1),           (0, 1),   # left, right
        (1, -1),  (1, 0),  (1, 1)    # down-left, down, down-right
    ]
    
    # Move first point
    for da, db in directions:
        new_a, new_b = a + da, b + db
        if 0 <= new_a <= grid_size and 0 <= new_b <= grid_size:
            neighbors.append([new_a, new_b, c, d])
    
    # Move second point
    for dc, dd in directions:
        new_c, new_d = c + dc, d + dd
        if 0 <= new_c <= grid_size and 0 <= new_d <= grid_size:
            neighbors.append([a, b, new_c, new_d])
    
    return neighbors

def dict_plan(plan, grid_size):
    
    size = grid_size+1
    initial = {i: 0 for i in range(size*size)}
    initial[coords_to_flat_index(plan[0],plan[1],size)] = 1
    initial[coords_to_flat_index(plan[2],plan[3],size)] = 1

    return initial


plan = [8,16,12,4]
Num_grid = 20
neighbors_4dir = get_plan_neighbors_4directional(plan,Num_grid)

yvalues_h = dict_plan(plan, Num_grid)
hubs = []
for k,y in yvalues_h.items():
    if y > 0:
        hubs.append(k)
        print(k)

for i in hubs:
    print(ColumnsName[i])
      

# print(f"Original plan: {plan}")
# print(f"4-directional neighbors: {len(neighbors_4dir)} total")

# # Show first few neighbors
# print("First 5 neighbors (4-directional):")
# for i, neighbor in enumerate(neighbors_4dir):
# 	print(f"  {neighbor}")

### Extended experiment

In [ ]:

# 20*20. 0:20 [8,16,12,4] 
# 20*50 [20,10,30,40]
start_time = time.time()
current_plan = [20,10,30,40]
# current_plan = [random.randint(0, 20),random.randint(0, 20),random.randint(0, 20),random.randint(0, 20) ]
print("Initial plan is: ", current_plan)
Num_grid = 50 # 20 50
best_plan = current_plan
best_score = compute_obj_value(current_plan, Num_grid)
isLocal = False
pool = PlanPool()
pool.add_plan(current_plan)

print("The current score is: ", best_score)


while not isLocal:
    isLocal = True
    # neighbors = get_plan_neighbors_4directional(plan,Num_grid)
    neighbors = get_plan_neighbors_8directional(current_plan,Num_grid)

    for i, neighbor in enumerate(neighbors):

        if pool.add_plan(neighbor):
            print("Catch a new plan, and the pool size is: ", pool.get_pool_size() )
            temp_score = compute_obj_value(neighbor, Num_grid)
            # print("Evaluating the plan: ", neighbor)
            if temp_score < best_score:
                print("We got a new plan. The current score is:", temp_score)
                isLocal = False
                best_plan = neighbor
                best_score = temp_score
                print("The best plan is:", best_plan)
        else:
            print("Catch a duplicated plan")
            
    current_plan = best_plan


print("The best plan is:", best_plan)

best_plan[0] = best_plan[0]+1
best_plan[1] = best_plan[1]+1
best_plan[2] = best_plan[2]+1
best_plan[3] = best_plan[3]+1
print("The best plan is:", best_plan)
print("The best score is:", best_score)


# 50*50 0:50 [20,10,30,40] ----- [21,11,31,41]

# initial = {i: 0 for i in range(2601)}
# initial[1030] = 1
# initial[1570] = 1

print("-------------- Elapsed %s seconds --------------" % (time.time() - start_time))




## SPSA

In [ ]:


def Bender_solution_trans(bender_solution,GridNumber):
    result = []
    for k in range(0,len(bender_solution),2):

        result.append(1/(GridNumber)*(bender_solution[k]-1))
        result.append(1/(GridNumber)*(bender_solution[k+1]-1))
    return np.float32(result)
        

def generate_sample(sample_size):

    column_index_name = [f'X1',f'X2',f'Y1',f'Y2']
    row_index_name = [f'Pair{i}' for i in range(1,sample_size+1)]
    sample_df = pd.DataFrame( index= row_index_name, columns=column_index_name)

    for i in range(len(sample_df)):

        sample_df.iloc[i] = np.random.uniform(0,1,4)

    return sample_df

def evaluate_obj(sample_df, solution):

    obj_value = 0
    for i in range(len(sample_df)):
        [X_1,X_2,Y_1,Y_2] = sample_df.iloc[i]
        temp = []
        for j in range(0,len(solution),2):
            P_1 = solution[j]
            P_2 = solution[j+1]
            temp.append(np.abs(X_1-P_1)+np.abs(X_2-P_2)+np.abs(Y_1-P_1)+np.abs(Y_2-P_2)) 

        obj_value += min(temp)
            
    return obj_value/len(sample_df)

def objective_variance(solution,sample_size):
    var_case = []
    for i in range(20):
        sample= generate_sample(sample_size)
        var_case.append(evaluate_obj(sample, solution))

    return variance(var_case)

def Reward_print_average(data):
    
    x = np.linspace(1, len(data),len(data) )
    # Plot the line plot using Seaborn's lineplot function
    seaborn.lineplot(x=x, y=data)
    plt.xlabel('Iteration')
    plt.ylabel('Objective value in kth iteration')
    return 1




def SPSA_optimization(initial_solution,sample_size,gavg,iter_number,step):
    optimal_solution = initial_solution
    lens_solution = len(optimal_solution)
    ## Automatic Gain Selection
    alpha = 0.602
    gamma=0.101
    eval_sample = generate_sample(sample_size)
    #var_sol = objective_variance(initial_solution,sample_size)
    var_sol = 0
    c = np.float32(max((var_sol /gavg) ** 0.5, .0001))
    A = 0.1*2000/(2*gavg) # 1000 is the expected number of loss evaluations per run? 125 iteration
    
    
    gbar = 0
    for _ in range(10):
        ghat = 0
        for j in range(gavg):
            delta = np.random.randint(0,2,lens_solution) * 2 - 1
            obj_plus = evaluate_obj(eval_sample,optimal_solution+delta*c)
            obj_min = evaluate_obj(eval_sample,optimal_solution-delta*c)
            ghat += (obj_plus - obj_min) / (2.0 * c * delta)
        gbar += np.abs(ghat/gavg)

    meanbar = np.mean(gbar)/10
    a = step*((A+1)**alpha)/meanbar

    print("A: ",A,"c: ",c,"a: ",a)

    ### Start iteration
    #start_time = time.time()
    k = 0
    flag = 1
    obj_track = []
    current_obj = evaluate_obj(eval_sample,optimal_solution)
    print("The initial objective function is", current_obj)
    old_obj = 0
    obj_track.append(current_obj)
    terminal_count = 0 
    unconverge_flag = 0

    while k < iter_number and flag:
        #print('========== Iteration ', k, ' ========== ')
        a_k = a/(A+1+k)**alpha
        c_k = c/(k+1)**gamma
        old_obj = current_obj
        g_k = 0
        for _ in range(gavg):
            delta = np.random.randint(0,2,lens_solution) * 2 - 1
            obj_plus = evaluate_obj(eval_sample,optimal_solution+delta*c_k)
            obj_min = evaluate_obj(eval_sample,optimal_solution-delta*c_k)
            g_k += (obj_plus - obj_min) / (2.0 * c_k * delta)

        g_k = g_k/gavg
        #print("the g_k is ", g_k)
        optimal_solution = optimal_solution - a_k * g_k
        current_obj = evaluate_obj(eval_sample,optimal_solution)
        #print("Current objective value is: ", current_obj)
        obj_track.append(current_obj)

        
        
        if abs(current_obj-old_obj) <= 1e-6:
            terminal_count += 1
        else:
            
            terminal_count = 0

        if terminal_count == 5:
            flag = 0        

        k += 1
        #print("------ Time Elapsed in the iteration %s seconds ------" % (time.time() - start_time))

    for item in optimal_solution:
            if item <= 0 or item >=1:
                unconverge_flag = 1
                #flag = 0  

    if obj_track[-1] >= obj_track[0]:
        unconverge_flag = 1
    
    print("The final objective function is", current_obj)


    return optimal_solution, obj_track, current_obj,k, unconverge_flag

In [ ]:
### SPSA with random start
pnumber = 10
initial_solution = np.random.uniform(0,1,2*pnumber)
#initial_solution= [0.4, 0.2, 0.8, 0.6, 0.3, 0.7]
print(initial_solution)
sample_size = 100000
gavg = 4
iter_number = 500
step = 0.1 # the initial desired magnitude of change in the theta elements
start_time = time.time()
optimal_solution, obj_track, current_obj,iter_SPSA,unconverge_flag= SPSA_optimization(initial_solution,sample_size,gavg,iter_number,step)

if unconverge_flag == 1:
    print("It is unconverge")
else:
    print("It is converge")

Reward_print_average(obj_track)
print(optimal_solution)
print(current_obj)
print("------ Time Elapsed %s seconds ------" % (time.time() - start_time))






In [ ]:
### SPSA with bender start
bender_solution = [11, 31,16, 11,21, 46,31, 21,41, 36,46, 11]

initial_solution = Bender_solution_trans(bender_solution,50)
print(initial_solution)
sample_size = 100000
gavg = 4
iter_number = 500
step = 0.01 # the initial desired magnitude of change in the theta elements
start_time = time.time()
optimal_solution_bender, obj_track,current_obj,iter_SPSA,unconverge_flag = SPSA_optimization(initial_solution,sample_size,gavg,iter_number,step)

if unconverge_flag == 1:
    print("It is unconverge")
else:
    print("It is converge")
    
Reward_print_average(obj_track)
print(optimal_solution_bender)
print(current_obj)
print("------ Time Elapsed %s seconds ------" % (time.time() - start_time))

### Print Result

In [ ]:
#### Running for random start

def print_random_start(pnumber,running_iterations,sample_size,step):
    
    running_time_queue = []
    optimal_solution_queue = [] 
    iteration_track = []
    gavg = 4
    iter_number = 500
    
    non_converge_count = 0 
    original_stdout = sys.stdout
    # Create a file name using the formatted date
    log_file_path = f"SPSA_random_{step}_{pnumber}_{sample_size}point.txt"
    print("Start the calculation of ", pnumber)

    with open(log_file_path, 'w') as f:
        sys.stdout = f
        for i in range(running_iterations):
            print("Start at the interation: ", i)
            initial_solution = np.random.uniform(0,1,2*pnumber)
            #initial_solution= [0.4, 0.2, 0.8, 0.6, 0.3, 0.7]
            #print(initial_solution)
            start_time = time.time()
            optimal_solution, obj_track, optimal_obj,iter_SPSA,unconverge_flag= SPSA_optimization(initial_solution,sample_size,gavg,iter_number,step)

            
            if iter_SPSA == iter_number or unconverge_flag == 1:
                print("Unconverge at the interation: ", i)
                non_converge_count +=1
            else:
                running_time_queue.append(time.time()-start_time)
                optimal_solution_queue.append(optimal_solution)
                iteration_track.append(iter_SPSA)
                

        print("the optimal solution queue is: \n", optimal_solution_queue)
        print("the running time queue is: \n", running_time_queue)
        print("the average running time is: \n", np.mean(running_time_queue))
        print("the std running time is: \n", np.std(running_time_queue))
        print("the iteration_track queue is: \n", iteration_track)
        print("the average iteration is: \n", np.mean(iteration_track))
        print("the std iteration is: \n", np.std(iteration_track))
        print("the number of unconverge case is: \n", non_converge_count)
        
        sys.stdout = original_stdout
    
    print(f"Print statements saved in the log file: {log_file_path}")


running_iterations = 20
sample_size = 5000
step = 0.05
for pnumber in [7,8,9,10]:
    print_random_start(pnumber,running_iterations,sample_size,step)


In [ ]:
def print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step):
    
    running_time_queue = []
    optimal_solution_queue = [] 
    iteration_track = []
    gavg = 4
    iter_number = 500
    non_converge_count = 0 
    initial_solution = Bender_solution_trans(bender_solution,20)
    original_stdout = sys.stdout
    # Create a file name using the formatted date
    log_file_path = f"SPSA_Bender_{step}_{pnumber}_{sample_size}point.txt"
    print("Start the calculation of ", pnumber)

    with open(log_file_path, 'w') as f:
        sys.stdout = f
        for i in range(running_iterations):
            start_time = time.time()
            
            optimal_solution, obj_track, optimal_obj,iter_SPSA,unconverge_flag= SPSA_optimization(initial_solution,sample_size,gavg,iter_number,step)
            
            if iter_SPSA == iter_number or unconverge_flag == 1:
                print("Unconverge at the interation: ", i)
                non_converge_count +=1
            else:
                running_time_queue.append(time.time()-start_time)
                optimal_solution_queue.append(optimal_solution)
                iteration_track.append(iter_SPSA)

        print("the optimal solution queue is: \n", optimal_solution_queue)
        print("the running time queue is: \n", running_time_queue)
        print("the average running time is: \n", np.mean(running_time_queue))
        print("the std running time is: \n", np.std(running_time_queue))
        print("the iteration_track queue is: \n", iteration_track)
        print("the average iteration is: \n", np.mean(iteration_track))
        print("the std iteration is: \n", np.std(iteration_track))
        print("the number of unconverge case is: \n", non_converge_count)
        
        sys.stdout = original_stdout
    
    print(f"Print statements saved in the log file: {log_file_path}")

running_iterations = 20
sample_size = 5000
step = 0.02

# # 2 hubs
# bender_solution = [9,15,13,7]
# pnumber = int(len(bender_solution)/2)
# print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)

# # 3 hubs
# bender_solution = [5, 9,11, 17,15, 7]
# pnumber = int(len(bender_solution)/2)
# print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)

# # 4 hubs
# bender_solution = [5, 13,9, 5,13, 17,17, 9]
# pnumber = int(len(bender_solution)/2)
# print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)


# # 5 hubs
# step = 0.005
# bender_solution = [5, 15,7, 5,11, 11,15, 17,17, 7]
# pnumber = int(len(bender_solution)/2)
# print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)

# step = 0.02

# 6 hubs
bender_solution = [5, 9,7, 17,9, 3,13, 13,17, 7,19, 17]
pnumber = int(len(bender_solution)/2)
print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)

# 7 hubs
bender_solution = [3, 7,5, 15,9, 3,11, 11,13, 19,17, 7,19, 15]
pnumber = int(len(bender_solution)/2)
print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)


# # 8 hubs
bender_solution = [3, 7,5, 17,9, 3,9, 13,13, 9,13, 19,17, 5,19, 15]
pnumber = int(len(bender_solution)/2)
print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)


# 9 hubs
bender_solution = [3, 9,5, 17,7, 3,9, 13,11, 7,13, 19,15, 11,17, 5,19, 15]
pnumber = int(len(bender_solution)/2)
print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)

# 10 hubs
bender_solution = [3, 13,5, 5,7, 19,9, 9,11, 3,11, 15,15, 11,17, 5,17, 19,19, 13]
pnumber = int(len(bender_solution)/2)
print_Bender_start(pnumber,running_iterations,sample_size,bender_solution,step)






In [ ]:
2+1280*9

### Evaluation


In [ ]:
### Generate evaluation sample
# eval_sample_size = 10000000
# eval_sample = generate_sample(eval_sample_size)

eval_sample = pd.read_csv("eval_sample.csv")
print(eval_sample)
eval_sample.drop(columns=eval_sample.columns[0], axis=1, inplace=True)
print(eval_sample)


In [ ]:
def post_evaluation(eval_solution,eval_sample,sample_size):
    
    eval_result = []
    p_number = int(len(eval_solution[0])/2)
    
    running_iterations = len(eval_solution)
    original_stdout = sys.stdout

    # Create a file name using the formatted date
    log_file_path = f"Evaluation_Bender_{p_number}_{sample_size}point.txt"
    print("Start the calculation of ", p_number)
    
    with open(log_file_path, 'w') as f:

       sys.stdout = f
       for i in range(running_iterations):
       
              temp_result = evaluate_obj(eval_sample,eval_solution[i])
              eval_result.append(temp_result)


       print("the eval_result is: \n", eval_result)
       print("the mean of obj value is: \n", np.mean(eval_result))
       print("the std of obj value is: \n", np.std(eval_result))
       sys.stdout = original_stdout

    print(f"Print statements saved in the log file: {log_file_path}")

    return eval_result


sample_size = 5000








## Bender 0.02 10point

locations =  [  [0.1380948 , 0.54300577, 0.17073592, 0.19137463, 0.25018355,
       0.82277826, 0.3566224 , 0.36397854, 0.52525092, 0.12775362,
       0.45976762, 0.69797402, 0.62624646, 0.48849567, 0.78954006,
       0.26469001, 0.695394  , 0.88182193, 0.87289543, 0.62554822]   ,   [0.14689368, 0.57034554, 0.19434501, 0.18305452, 0.24026943,
       0.86392238, 0.35647417, 0.34755699, 0.58701154, 0.12349929,
       0.47602889, 0.71902308, 0.61417728, 0.47944766, 0.80383636,
       0.27580182, 0.70223485, 0.8390653 , 0.87482234, 0.62211883]   ,   [0.12573163, 0.56555334, 0.20029537, 0.21272848, 0.27896981,
       0.81403471, 0.37471672, 0.38661761, 0.55992288, 0.13976829,
       0.48741354, 0.66072488, 0.653158  , 0.50565522, 0.82445521,
       0.28833036, 0.72660537, 0.84146538, 0.89392905, 0.61989014]   ,   [0.12217476, 0.57526746, 0.2043288 , 0.21984624, 0.25921282,
       0.87374299, 0.35711188, 0.39460765, 0.51167908, 0.12564659,
       0.45310282, 0.68096404, 0.62507182, 0.46916225, 0.79815825,
       0.26251152, 0.69126223, 0.8326645 , 0.87538084, 0.58686595]   ,   [0.14361868, 0.54841782, 0.21282894, 0.208506  , 0.28560083,
       0.85279639, 0.37493155, 0.37032672, 0.52864576, 0.13154456,
       0.4900173 , 0.64179488, 0.66718217, 0.44082154, 0.81910264,
       0.24403675, 0.70713138, 0.80827241, 0.89356064, 0.56938757]   ,   [0.11991795, 0.56040969, 0.1898449 , 0.21894867, 0.28049009,
       0.81520065, 0.34411878, 0.38460061, 0.50720515, 0.12772702,
       0.47808498, 0.66963017, 0.62632447, 0.48654848, 0.79627257,
       0.25125963, 0.69562619, 0.85739513, 0.87438082, 0.61426636]   ,   [0.1313654 , 0.5426283 , 0.21025874, 0.21592052, 0.27255339,
       0.85557337, 0.36691864, 0.36096258, 0.54418333, 0.12827569,
       0.43929224, 0.68327504, 0.61348902, 0.48328882, 0.81167142,
       0.2753732 , 0.72160537, 0.81222745, 0.8762653 , 0.62838524]   ,   [0.15415557, 0.61125143, 0.21653505, 0.21001208, 0.29666023,
       0.87390371, 0.39504382, 0.39295188, 0.56827807, 0.12552395,
       0.48849673, 0.68841833, 0.64893104, 0.49934195, 0.80367716,
       0.27489685, 0.7324034 , 0.83491478, 0.89339863, 0.5865177 ]   ,   [0.17571163, 0.65109347, 0.13642088, 0.27521851, 0.33809598,
       0.878233  , 0.34567466, 0.39934966, 0.4869699 , 0.13609914,
       0.51249521, 0.68839961, 0.640628  , 0.46307388, 0.77722703,
       0.24330313, 0.74618295, 0.80814469, 0.87399378, 0.56087392]   ,   [0.13158168, 0.58092605, 0.21271083, 0.21651312, 0.26737409,
       0.84415523, 0.36698566, 0.40297387, 0.54050043, 0.13072147,
       0.49722499, 0.71377214, 0.64588361, 0.51767219, 0.80364507,
       0.29950568, 0.69748171, 0.88363119, 0.88264896, 0.66573021]   ,   [0.1482844 , 0.54292987, 0.20196269, 0.19307002, 0.30513197,
       0.82428193, 0.38739772, 0.34390565, 0.54502839, 0.11006704,
       0.48647106, 0.67075092, 0.64014112, 0.43266217, 0.80875436,
       0.25197675, 0.72388002, 0.86032664, 0.86800428, 0.58534738]   ,   [0.14361578, 0.57162416, 0.21364727, 0.21414424, 0.29664041,
       0.82973562, 0.40847581, 0.37287352, 0.54481889, 0.13750225,
       0.51322742, 0.67933599, 0.65334031, 0.49434186, 0.81276335,
       0.28526587, 0.74337473, 0.85497063, 0.8916564 , 0.5772497 ]   ,   [0.13019582, 0.59285536, 0.20209631, 0.21156998, 0.26394504,
       0.8562898 , 0.36057455, 0.39364408, 0.51596684, 0.12886575,
       0.47674987, 0.67370038, 0.61879063, 0.47687321, 0.79556025,
       0.27349403, 0.67400523, 0.82885689, 0.88205887, 0.62818099]   ,   [0.14222906, 0.58969027, 0.21043595, 0.21503003, 0.28188427,
       0.8147799 , 0.37294882, 0.37558616, 0.52217147, 0.10567101,
       0.48055884, 0.67990023, 0.6172369 , 0.45644265, 0.80091031,
       0.25043503, 0.73795848, 0.85566969, 0.87389007, 0.58929956]   ,   [0.16154832, 0.55744115, 0.23315506, 0.19133514, 0.26800815,
       0.84790721, 0.38757568, 0.3554337 , 0.56937718, 0.11470458,
       0.47148535, 0.67120173, 0.63420567, 0.48948684, 0.79643755,
       0.25369662, 0.70709343, 0.82201384, 0.87144174, 0.62749956]   ,   [0.11409961, 0.49692041, 0.26539756, 0.24876617, 0.24763099,
       0.81476046, 0.40049936, 0.52667548, 0.53177784, 0.12983007,
       0.55465735, 0.7472527 , 0.63794548, 0.38435301, 0.88162875,
       0.25522666, 0.83262406, 0.87879541, 0.77768887, 0.59973525]   ,   [0.12763489, 0.58186924, 0.19483494, 0.2080971 , 0.28469945,
       0.80268192, 0.3641919 , 0.36854256, 0.55633747, 0.13450933,
       0.48637772, 0.65561122, 0.66987322, 0.44712027, 0.82201551,
       0.24855456, 0.68628462, 0.86630407, 0.86840267, 0.60546713]   ,   [0.13157254, 0.55901297, 0.19742479, 0.19454167, 0.24887096,
       0.82942368, 0.34217399, 0.38147017, 0.54818759, 0.11975241,
       0.47293977, 0.65823221, 0.62616263, 0.49105548, 0.79776561,
       0.29157231, 0.70078733, 0.86741789, 0.84904396, 0.67232968]   ,   [0.13853345, 0.62486206, 0.21654465, 0.24582682, 0.31198495,
       0.83332679, 0.40067079, 0.42003167, 0.54286324, 0.13454978,
       0.53014584, 0.67190545, 0.67725985, 0.48906826, 0.83083499,
       0.26842173, 0.74532889, 0.86951113, 0.89037545, 0.59719111]   ,   [0.1807041 , 0.56541828, 0.1323144 , 0.21077292, 0.26500867,
       0.86157537, 0.35164724, 0.35489325, 0.49214081, 0.12479642,
       0.47510043, 0.68771421, 0.60515484, 0.43524016, 0.78460603,
       0.22840391, 0.72019875, 0.829046  , 0.86314297, 0.57965637]   ]
post_evaluation(locations,eval_sample,sample_size    )   




